# PCA con autovalores: comprimir una cara en pocos números

**Material complementario avanzado — Laboratorio de Tecnologías de la Imagen Digital**

**IFTS 24 - 2026**

---

> **Ubicación sugerida:** este es un módulo optativo pensado para **más adelante en la cursada**, después de cerrar el capítulo 4 (detección de rostros). No hace falta resolverlo en las primeras semanas — requiere tener asentada la idea de "imagen como vector de píxeles" que se termina de ver en los capítulos 3 y 4.

## Objetivo

Usar autovalores y autovectores para resumir un conjunto de imágenes de caras en unos pocos números, y reconstruirlas con distinta cantidad de información.

## Al terminar este material van a poder:

1. Explicar qué es un autovector y un autovalor con un ejemplo simple.
2. Explicar qué es la matriz de covarianza de un conjunto de imágenes.
3. Aplanar (flatten) un conjunto de imágenes en una tabla de datos.
4. Calcular autovalores y autovectores de la matriz de covarianza.
5. Interpretar cada autovector como una "componente principal" y visualizarlo como imagen.
6. Reducir la dimensionalidad de un conjunto de imágenes con PCA y reconstruirlas con distinta cantidad de componentes.
7. Usar la varianza explicada para decidir cuántas componentes conservar.

## Microglosario

| Término | Significado en lenguaje llano |
|---|---|
| Autovector | Un vector que una matriz solo estira o achica, sin cambiarle la dirección. |
| Autovalor | El número por el que la matriz estira (o achica) a su autovector. |
| Matriz de covarianza | Una tabla que mide cuánto varían juntas las variables de un conjunto de datos — acá, los píxeles. |
| Componente principal | Cada autovector de la matriz de covarianza; representa una dirección importante de variación en los datos. |
| Dimensionalidad | La cantidad de números necesarios para describir cada dato — acá, la cantidad de píxeles de cada imagen. |
| Reducción de dimensionalidad | Resumir cada dato con muchos menos números, conservando la información más importante. |
| Varianza explicada | Qué porción de la variación total de los datos "explica" cada componente principal. |

## Una hoja de goma con una dirección especial

En el cuaderno de transformaciones lineales vimos que multiplicar una matriz por un vector es como estirar una hoja de goma: casi todas las flechas cambian de dirección.

Pero para casi cualquier matriz hay una o dos direcciones especiales que **no se tuercen**: solo se hacen más largas o más cortas, sin girar. Esas direcciones son los autovectores, y cuánto se estiran es el autovalor.

**Dónde vive esto en el mundo real:** cuando tenemos muchísimos datos con muchísimas variables (como una imagen, donde cada píxel es una variable), los autovectores de la matriz de covarianza nos dicen cuáles son las "direcciones" en las que más varían los datos. Es la base matemática detrás de técnicas de compresión de imágenes y de las primeras técnicas de reconocimiento facial automático (conocidas como *eigenfaces*), una especie de identikit matemático: en vez de armar una cara combinando nariz, ojos y boca, se arma combinando estas "caras fantasma".

## 1. Qué es un autovector

Empecemos con una matriz simple para ver la idea en acción.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("✓ Librerías cargadas correctamente.")

In [ ]:
matriz_simple = np.array([
    [2, 0],
    [0, 0.5],
])

vector_autovector = np.array([1, 0])
vector_cualquiera = np.array([1, 1])

resultado_autovector = matriz_simple @ vector_autovector
resultado_cualquiera = matriz_simple @ vector_cualquiera

print("✓ Comparación de vectores")
print("Vector (1, 0) transformado:", resultado_autovector, "→ misma dirección, solo se estiró")
print("Vector (1, 1) transformado:", resultado_cualquiera, "→ cambió de dirección")

`(1, 0)` es un autovector de esta matriz, con autovalor 2: se estiró al doble sin girar. `(1, 1)` no lo es, porque cambió de dirección (ya no es un múltiplo de `(1, 1)`).

NumPy puede encontrar automáticamente todos los autovectores y autovalores de una matriz con `np.linalg.eig`.

In [ ]:
autovalores_ejemplo, autovectores_ejemplo = np.linalg.eig(matriz_simple)

print("✓ Autovalores y autovectores calculados")
print("Autovalores:", autovalores_ejemplo)
print("Autovectores (en columnas):\n", autovectores_ejemplo)

### Para pensar

- ¿Los autovalores que encontró NumPy coinciden con lo que ya sabíamos de esta matriz?
- ¿Qué pasaría con los autovectores si la matriz fuera `[[3, 0], [0, 3]]`? Prueben cambiar `matriz_simple` y volver a correr las celdas.

## 2. De autovectores a caras: cargamos un dataset

Vamos a trabajar con un conjunto de 400 fotos de caras (el dataset Olivetti, un clásico para este tipo de ejercicios). La primera vez que corren esta celda, `scikit-learn` descarga el dataset — pesa pocos MB y puede tardar unos segundos.

In [ ]:
from sklearn.datasets import fetch_olivetti_faces

datos_caras = fetch_olivetti_faces()
imagenes = datos_caras.images

alto, ancho = imagenes[0].shape
cantidad_imagenes = imagenes.shape[0]

print("✓ Dataset de caras cargado.")
print("Cantidad de imágenes:", cantidad_imagenes)
print("Tamaño de cada imagen:", alto, "x", ancho, "píxeles")

In [ ]:
figura, ejes = plt.subplots(4, 4, figsize=(8, 8))

for fila in range(4):
    for columna in range(4):
        indice = fila * 4 + columna
        eje = ejes[fila, columna]
        eje.imshow(imagenes[indice], cmap="gray")
        eje.axis("off")

plt.tight_layout()
plt.show()

print("✓ Primeras 16 caras del dataset.")

### Para pensar

- Mirando estas 16 caras, ¿qué tienen en común? ¿En qué se diferencian?

## 3. Aplanamos las imágenes

Cuando trabajamos con una imagen sola, tenerla como una grilla de alto x ancho es cómodo para visualizarla. Pero para comparar 400 imágenes entre sí como si fueran puntos de datos, conviene convertir cada imagen en un único vector de números: aplanarla (flatten).

Cada imagen de 64x64 se convierte en un vector de 64*64 = 4096 números.

In [ ]:
imagenes_aplanadas = imagenes.reshape(cantidad_imagenes, alto * ancho)

print("✓ Imágenes aplanadas.")
print("Forma de la tabla de datos:", imagenes_aplanadas.shape)
print("(400 caras, cada una descrita con 4096 números)")

## 4. Centramos los datos

Antes de calcular la matriz de covarianza, restamos a cada imagen la "cara promedio" del dataset. Esto es un paso estándar de PCA: nos interesa cómo varía cada cara respecto del promedio, no los valores absolutos de brillo.

In [ ]:
cara_promedio = imagenes_aplanadas.mean(axis=0)
datos_centrados = imagenes_aplanadas - cara_promedio

print("✓ Datos centrados.")

plt.figure(figsize=(4, 4))
plt.imshow(cara_promedio.reshape(alto, ancho), cmap="gray")
plt.title("Cara promedio del dataset")
plt.axis("off")
plt.show()

## 5. La matriz de covarianza

La matriz de covarianza mide cuánto varía cada píxel en relación con cada otro píxel, a lo largo de las 400 caras. Como tenemos 4096 píxeles (variables), la matriz de covarianza es de 4096x4096. Este cálculo puede tardar unos segundos.

In [ ]:
cantidad_observaciones = imagenes_aplanadas.shape[0]

matriz_covarianza = (datos_centrados.T @ datos_centrados) / (cantidad_observaciones - 1)

print("✓ Matriz de covarianza calculada.")
print("Forma:", matriz_covarianza.shape)

### Para pensar

- ¿Por qué la matriz de covarianza es de 4096x4096 y no de 400x400?

## 6. Autovalores y autovectores de la covarianza

Calcular los 4096 autovalores de esta matriz sería muy costoso, y la mayoría no aportan información relevante. Por eso usamos `scipy.sparse.linalg.eigsh`, que calcula solo los `k` autovalores más grandes — los que más varianza explican. Este cálculo también puede tardar unos segundos.

In [ ]:
import scipy.sparse.linalg

cantidad_componentes = 60

autovalores, autovectores = scipy.sparse.linalg.eigsh(matriz_covarianza, k=cantidad_componentes)

# eigsh los devuelve de menor a mayor; dados vuelta para tener el más importante primero
autovalores = autovalores[::-1]
autovectores = autovectores[:, ::-1]

print("✓ Autovalores y autovectores calculados.")
print("Los 5 autovalores más grandes:", autovalores[:5])

## 7. Mirando las componentes principales como imágenes

Cada autovector tiene 4096 números — el mismo tamaño que una cara aplanada. Si lo volvemos a acomodar en una grilla de 64x64, podemos verlo como una imagen. Estas imágenes "fantasma" son las llamadas *eigenfaces*: cada una captura un patrón de variación entre las caras del dataset.

In [ ]:
figura, ejes = plt.subplots(4, 4, figsize=(10, 10))

for fila in range(4):
    for columna in range(4):
        indice = fila * 4 + columna
        componente = autovectores[:, indice].reshape(alto, ancho)
        eje = ejes[fila, columna]
        eje.imshow(componente, cmap="gray")
        eje.set_title(f"Componente {indice + 1}")
        eje.axis("off")

plt.tight_layout()
plt.show()

print("✓ Primeras 16 componentes principales.")

### Para pensar

- Las primeras componentes suelen capturar patrones generales (luz y sombra, forma general de la cara), y las siguientes, detalles más finos. ¿Se nota esa diferencia entre la componente 1 y la componente 16?

## 8. Reducimos la dimensionalidad

Para reducir cada cara a solo `k` números, multiplicamos los datos centrados por las primeras `k` columnas de autovectores. Primero lo hacemos paso a paso para `k=2`, y después lo encapsulamos en una función.

In [ ]:
cantidad_componentes_reducidas = 2

V = autovectores[:, :cantidad_componentes_reducidas]
datos_reducidos = datos_centrados @ V

print("✓ Datos reducidos a", cantidad_componentes_reducidas, "componentes.")
print("Forma original:", datos_centrados.shape)
print("Forma reducida:", datos_reducidos.shape)

In [ ]:
def reducir_con_pca(datos_centrados, autovectores, cantidad_componentes):
    V = autovectores[:, :cantidad_componentes]
    datos_reducidos = datos_centrados @ V
    return datos_reducidos

print("✓ Función reducir_con_pca creada.")

Con solo 2 números por cara, podemos graficar las 400 caras como puntos en un plano.

In [ ]:
plt.figure(figsize=(10, 10))
plt.scatter(datos_reducidos[:, 0], datos_reducidos[:, 1], alpha=0.6)

for indice in range(cantidad_imagenes):
    plt.text(datos_reducidos[indice, 0], datos_reducidos[indice, 1], str(indice), fontsize=8)

plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title("Las 400 caras resumidas en apenas 2 números")
plt.show()

print("✓ Gráfico generado.")

### Para pensar

- Elijan dos números de caras que aparezcan cerca en el gráfico y otros dos que aparezcan lejos. Múestrenlas con `imagenes[indice]` y comprueben si las cercanas se parecen más entre sí que las lejanas.

In [ ]:
# ─── Espacio de práctica ──────────────────────────────────────────────────
#
# 1. Elijan dos índices de caras que estén cerca en el gráfico anterior
# 2. Elijan dos índices de caras que estén lejos
# 3. Muestren las cuatro caras con plt.imshow(imagenes[indice], cmap="gray")
# 4. Comparen: ¿las cercanas se parecen más entre sí?
#
# ─────────────────────────────────────────────────────────────────────────


## 9. Reconstruir las imágenes

Reducir dimensiones pierde información. La pregunta natural es: ¿cuánta información hace falta conservar para que la reconstrucción siga siendo reconocible? Para reconstruir, hacemos el camino inverso: multiplicamos los datos reducidos por los autovectores traspuestos, y le sumamos de nuevo la cara promedio.

In [ ]:
def reconstruir_desde_pca(datos_reducidos, autovectores, cara_promedio, alto, ancho):
    cantidad_componentes = datos_reducidos.shape[1]
    datos_reconstruidos = datos_reducidos @ autovectores[:, :cantidad_componentes].T
    imagenes_reconstruidas = datos_reconstruidos + cara_promedio
    return imagenes_reconstruidas.reshape(-1, alto, ancho)

print("✓ Función reconstruir_desde_pca creada.")

In [ ]:
indice_cara_elegida = 21
cantidades_a_probar = [1, 5, 10, 20, 60]

figura, ejes = plt.subplots(1, len(cantidades_a_probar) + 1, figsize=(18, 4))

ejes[0].imshow(imagenes[indice_cara_elegida], cmap="gray")
ejes[0].set_title("Original")
ejes[0].axis("off")

for posicion, cantidad in enumerate(cantidades_a_probar):
    datos_reducidos_prueba = reducir_con_pca(datos_centrados, autovectores, cantidad)
    imagenes_reconstruidas = reconstruir_desde_pca(datos_reducidos_prueba, autovectores, cara_promedio, alto, ancho)

    eje = ejes[posicion + 1]
    eje.imshow(imagenes_reconstruidas[indice_cara_elegida], cmap="gray")
    eje.set_title(f"{cantidad} componentes")
    eje.axis("off")

plt.tight_layout()
plt.show()

print("✓ Reconstrucciones generadas.")

### Para pensar

- ¿A partir de cuántas componentes la cara reconstruida les resulta reconocible?
- Recuerden que cada cara original tiene 4096 números. Con 20 componentes, ¿cuántas veces menos información estamos usando?

## 10. Varianza explicada

En vez de elegir la cantidad de componentes "a ojo", podemos mirar qué porcentaje de la variación total de los datos explica cada una.

Hay un detalle importante acá: solo calculamos los `cantidad_componentes` autovalores más grandes, no los cerca de 400 que existen en total (recuerden que tenemos 400 caras). Por eso **no podemos** usar la suma de esos pocos autovalores como "el total" — nos daría siempre 100%, sin importar cuántas componentes hayamos calculado, lo cual sería engañoso.

La varianza total real es la suma de las varianzas de los 4096 píxeles, y se puede calcular directamente como la **traza** de la matriz de covarianza (la suma de su diagonal) — sin necesidad de encontrar todos sus autovalores.

In [ ]:
varianza_total = np.trace(matriz_covarianza)

varianza_explicada = autovalores / varianza_total
varianza_acumulada = np.cumsum(varianza_explicada)

plt.figure(figsize=(8, 5))
plt.plot(range(1, cantidad_componentes + 1), varianza_acumulada)
plt.axhline(0.95, color="red", linestyle="--", label="95% de la varianza")
plt.xlabel("Cantidad de componentes")
plt.ylabel("Varianza acumulada explicada")
plt.legend()
plt.show()

print("✓ Gráfico de varianza explicada generado.")
print(f"Con las {cantidad_componentes} componentes calculadas, explicamos el {varianza_acumulada[-1]:.1%} de la varianza total real de los datos.")

In [ ]:
componentes_para_95 = np.argmax(varianza_acumulada >= 0.95) + 1

if varianza_acumulada[-1] >= 0.95:
    print("Componentes necesarias para explicar el 95% de la varianza:", componentes_para_95)
else:
    print("Con", cantidad_componentes, "componentes todavía no llegamos al 95%.")
    print("Para encontrar el número exacto, aumenten 'cantidad_componentes' en la sección 6 y vuelvan a correr desde ahí.")

### Para pensar

- ¿El número de componentes necesarias para el 95% de la varianza es mayor o menor de lo que esperaban, comparado con los 4096 píxeles originales?

## Cierre

En este cuaderno usamos autovalores y autovectores para resumir 400 caras — cada una descrita originalmente con 4096 números — en unas pocas decenas de componentes, y pudimos reconstruirlas con distinto nivel de detalle según cuántas conservamos.

Esta misma idea (resumir una imagen en pocos números que capturan lo más importante) es un antecedente directo de cómo las redes neuronales modernas de visión por computadora representan imágenes internamente: en vez de trabajar con cada píxel por separado, aprenden a resumir la imagen en un puñado de números que capturan lo esencial.

## Actividad de producción (optativa)

1. Elijan otro índice de cara distinto al 21 y repitan la comparación de reconstrucciones con distintas cantidades de componentes.
2. Prueben aumentar `cantidad_componentes` a 100 en la sección 6 (va a tardar más en calcular) y vean cuánto sube el porcentaje de varianza explicada.
3. Miren la componente principal número 1 como imagen: ¿qué tipo de variación general de las caras les parece que está capturando?

In [ ]:
# ─── Espacio de práctica ──────────────────────────────────────────────────
#
# 1. Elijan otro índice de cara y repitan la comparación de reconstrucciones
# 2. Prueben cambiar la cantidad de componentes calculadas en la sección 6
# 3. Anoten sus observaciones sobre la componente principal número 1
#
# ─────────────────────────────────────────────────────────────────────────
